In [ ]:
# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('../..'))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

choose metrics to predict shot distribution:
- choice after ball reception (pass + shot/carry)
- distance/angle of pass and carry
- key pass/shot
- average reception, average release

for determining the choice after ball reception, need to see event dataframe for a match/game and see how timings work

In [ ]:
PASS_OUTCOME_QUERY = """
CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50' AND ("50_50" LIKE '%Won%'))) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Pass'
GROUP BY player_id;
"""

CARRY_OUTCOME_QUERY = """
CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50' AND ("50_50" LIKE '%Won%'))) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Complete Carry'
GROUP BY player_id;
"""

SHOT_OUTCOME_QUERY = """
CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50' AND ("50_50" LIKE '%Won%'))) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Shot'
GROUP BY player_id;
"""


In [ ]:
MAIN_QUERY = """
CREATE TEMP TABLE pass_distance AS SELECT
    player_id,
    first(player) AS player,
    avg(pass_length) AS average_pass_length,
    avg(pass_angle) AS average_pass_angle,
    count(*) AS pass_total
FROM events
WHERE
    (
        (type = 'Pass') AND
        (position <> 'Goalkeeper')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )
GROUP BY player_id, type;

CREATE TEMP TABLE carry_distance AS SELECT
    player_id,
    first(player) AS player,
    avg(end_x - x) AS average_x_displacement,
    avg(end_y - y) AS average_y_displacement,
    count(*) AS carry_total
FROM events
WHERE
    (
        (type = 'Complete Carry')
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    )  
GROUP BY player_id;

CREATE TEMP TABLE player_rel AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    CASE
        WHEN type = 'Complete Carry' THEN end_x
        ELSE x
        END AS x,
    CASE
        WHEN type = 'Complete Carry' THEN end_y
        ELSE y
        END AS y
FROM
    events
WHERE
    (
        (type = 'Pass') OR
        (type = 'Shot') OR
        ((type = 'Complete Carry') AND (carry_end_outcome = 'Other'))
    ) 
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    );

CREATE TEMP TABLE player_rel_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rel_x,
    avg(y) AS avg_rel_y,
    avg(40 - ABS(y - 40)) AS avg_normrel_y,
    count(*) AS rel_total
FROM player_rel
GROUP BY player_id;

CREATE TEMP TABLE player_rec AS SELECT
    player_id,
    match_id,
    team_id,
    player,
    position,
    type,
    match_seconds,
    teamsheet,
    mandown,
    play_pattern,
    x,
    y,
FROM
    events
WHERE 
    (
        (type = 'Ball Receipt*') OR
        (type = 'Ball Recovery') OR
        ((type = '50/50') AND ("50_50" LIKE '%Won%')) OR
        ((type = 'Duel') AND (duel_outcome = 'Won')) OR
        ((type = 'Interception') AND (interception_outcome = 'Won')) OR
        ((type = 'Pass') AND (pass_type = 'Recovery')) OR
        ((type = 'Shot') AND (shot_type = 'Recovery'))
    )
    AND (mandown = False)
    AND
    (
        (play_pattern = 'Regular Play') OR
        (play_pattern = 'From Keeper') OR
        (play_pattern = 'From Counter') OR
        (play_pattern = 'From Throw In')
    );

CREATE TEMP TABLE player_rec_avg AS SELECT
    player_id,
    first(player) AS player,
    avg(x) AS avg_rec_x,
    avg(y) AS avg_rec_y,
    avg(40 - ABS(y - 40)) AS avg_normrec_y,
    count(*) AS rec_total
FROM player_rec
GROUP BY player_id;

CREATE TEMP TABLE key_pass AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS key_pass_total
FROM
    events
WHERE
    (
        (type = 'Pass') AND
        (pass_shot_assist)
    )
GROUP BY player_id;

CREATE TEMP TABLE shot_count AS SELECT
    player_id,
    first(player) AS player,
    count(*) AS shot_total
FROM events
WHERE type = 'Shot' 
GROUP BY player_id;

CREATE TEMP TABLE reception AS
    SELECT
        player_id,
        match_id,
        team_id,
        player,
        type,
        match_seconds,
        play_pattern,
        x,
        y,
    FROM
        events
    WHERE
        (
            (type = 'Ball Receipt*') OR
            (type = 'Ball Recovery') OR
            ((type = '50/50' AND ("50_50" LIKE '%Won%'))) OR
            ((type = 'Duel') AND (duel_outcome = 'Won')) OR
            ((type = 'Interception') AND (interception_outcome = 'Won'))
        ) 
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE outcome AS
    SELECT
        player_id,
        match_id,
        team_id,
        type,
        match_seconds
    FROM
        events
    WHERE
        (
            (type = 'Pass') OR
            (type = 'Shot') OR
            (type = 'Complete Carry')
        )
    ORDER BY
        match_id, team_id, match_seconds;

CREATE TEMP TABLE reception_outcomes AS
SELECT
    l.player_id as player_id,
    l.match_id as match_id,
    l.team_id as team_id,
    l.player as player,
    l.type as type,
    l.match_seconds as match_seconds,
    l.play_pattern as play_pattern,
    l.x as x,
    l.y as y,
    r.type as reception_outcome
FROM reception l LEFT JOIN outcome r ON
    l.player_id = r.player_id AND
    l.match_id = r.match_id AND
    l.team_id = r.team_id AND
    l.match_seconds = r.match_seconds;

CREATE TEMP TABLE outcome_pass_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Pass'
GROUP BY player_id;

CREATE TEMP TABLE outcome_carry_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Complete Carry'
GROUP BY player_id;

CREATE TEMP TABLE outcome_shot_count AS
SELECT
    player_id,
    first(player) as player,
    count(*) as outcome_count
FROM reception_outcomes
WHERE reception_outcome = 'Shot'
GROUP BY player_id;

SELECT
    l.player_id AS player_id,
    l.player AS player,
    l.average_pass_length AS average_pass_length,
    l.average_pass_angle AS average_pass_angle,
    l.pass_total AS pass_total,
    r.average_x_displacement AS average_carry_x_displacement,
    r.average_y_displacement AS average_carry_y_displacement,
    r.carry_total AS carry_total,
    a.avg_rel_x AS average_release_x,
    a.avg_rel_y AS average_release_y,
    a.avg_normrel_y AS average_norm_release_y,
    a.rel_total AS rel_total,
    b.avg_rec_x AS average_reception_x,
    b.avg_rec_y AS average_reception_y,
    b.avg_normrec_y AS average_norm_reception_y,
    b.rec_total AS rec_total,
    (c.key_pass_total / (c.key_pass_total + d.shot_total)) AS key_pass_shot_ratio,
    c.key_pass_total AS key_pass_total,
    d.shot_total AS shot_total,
    (oc.outcome_count/(op.outcome_count + oc.outcome_count + osh.outcome_count)) AS carry_shotpass_ratio,
    op.outcome_count AS outcome_pass_total,
    oc.outcome_count AS outcome_carry_total,
    osh.outcome_count AS outcome_shot_total
FROM pass_distance l
INNER JOIN carry_distance r ON
    l.player_id = r.player_id AND
    l.player = r.player
INNER JOIN player_rel_avg a ON
    l.player_id = a.player_id AND
    l.player = a.player
INNER JOIN player_rec_avg b ON
    l.player_id = b.player_id AND
    l.player = b.player
INNER JOIN key_pass c ON
    l.player_id = c.player_id AND
    l.player = c.player
INNER JOIN shot_count d ON
    l.player_id = d.player_id AND
    l.player = d.player
INNER JOIN outcome_pass_count op ON
    l.player_id = op.player_id AND
    l.player = op.player
INNER JOIN outcome_carry_count oc ON
    l.player_id = oc.player_id AND
    l.player = oc.player
INNER JOIN outcome_shot_count osh ON
    l.player_id = osh.player_id AND
    l.player = osh.player
WHERE
    (a.rel_total >= 50 AND
    b.rec_total >= 50);
"""

In [ ]:
DB_PATH = '/home/kishor/Projects/ShotDist/shotdistributionpredictor/data/events.db'
con = duckdb.connect(DB_PATH, read_only = True)
con.execute('PRAGMA threads = 10')
player_df = con.execute(MAIN_QUERY).df()
con.close()

In [ ]:
feature_cols = [
    "average_pass_length",
    "average_pass_angle",
    "average_carry_x_displacement",
    "average_carry_y_displacement",
    "average_release_x",
    "average_norm_release_y",
    "average_reception_x",
    "average_norm_reception_y",
    "key_pass_shot_ratio",
    "carry_shotpass_ratio"
]

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

valid = np.isfinite(player_df[feature_cols]).all(axis=1)
X_scaled = StandardScaler().fit_transform(player_df.loc[valid, feature_cols])

linkage_matrix = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=40,
    show_contracted=True,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)
ax.set_title("Player hierarchy (Ward linkage)")
ax.set_xlabel("Player groups (truncated to the last 40 merges)")
ax.set_ylabel("Ward distance")
plt.tight_layout()
plt.show()

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

valid = np.isfinite(player_df[feature_cols]).all(axis=1)
X_scaled = StandardScaler().fit_transform(player_df.loc[valid, feature_cols])

linkage_matrix = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(
    linkage_matrix,
    truncate_mode="lastp",
    p=40,
    show_contracted=True,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)
ax.set_title("Player hierarchy (Ward linkage)")
ax.set_xlabel("Player groups (truncated to the last 40 merges)")
ax.set_ylabel("Ward distance")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering

n_clusters = 15
valid = np.isfinite(player_df[feature_cols]).all(axis=1)
X_scaled = StandardScaler().fit_transform(player_df.loc[valid, feature_cols])

clustered_df = player_df.copy()
clustered_df["cluster"] = pd.Series(pd.NA, index=clustered_df.index, dtype="Int64")
model = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
clustered_df.loc[valid, "cluster"] = model.fit_predict(X_scaled)

map_cols = [
    "average_reception_x",
    "average_reception_y",
    "average_release_x",
    "average_release_y",
]
pitch = Pitch(
    pitch_type="statsbomb",
    pitch_color="white",
    line_color="#333333",
    axis=True,
    label=True,
    tick=True,
)
ncols = 3
nrows = (n_clusters + ncols - 1) // ncols
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 4 * nrows))
axes = axes.flatten()
colours = plt.get_cmap("turbo", n_clusters)

for cluster_id, ax in enumerate(axes[:n_clusters]):
    pitch.draw(ax=ax)
    cluster_rows = clustered_df.loc[clustered_df["cluster"] == cluster_id]
    map_valid = np.isfinite(cluster_rows[map_cols]).all(axis=1)
    map_rows = cluster_rows.loc[map_valid]
    colour = colours(cluster_id)

    pitch.lines(
        map_rows["average_reception_x"],
        map_rows["average_reception_y"],
        map_rows["average_release_x"],
        map_rows["average_release_y"],
        color=colour,
        alpha=0.35,
        lw=0.7,
        ax=ax,
    )
    pitch.scatter(
        map_rows["average_reception_x"],
        map_rows["average_reception_y"],
        color=colour,
        marker="o",
        s=24,
        ax=ax,
    )
    pitch.scatter(
        map_rows["average_release_x"],
        map_rows["average_release_y"],
        color=colour,
        marker="^",
        s=28,
        ax=ax,
    )
    ax.set_title(f"Cluster {cluster_id} (n={len(cluster_rows)})")

for ax in axes[n_clusters:]:
    ax.set_visible(False)

fig.suptitle("Agglomerative player clusters: reception to release", y=1.01)
plt.tight_layout()
plt.show()